# Missing Values in Pandas

Explore detection, profiling, deletion, imputation, placeholders, group-based filling, and missingness indicators.

In [1]:
import pandas as pd
import numpy as np

df = pd.DataFrame({
    'name': ['Asha', 'Ravi', 'Meera', 'Kabir', None],
    'age': [21, np.nan, 23, 22, 20],
    'city': ['Indore', 'Pune', None, 'Delhi', 'Pune'],
    'score': [88, 74, np.nan, 91, 85]
})
df

,name,age,city,score
0,Asha,21.0,Indore,88.0
1,Ravi,NaN,Pune,74.0
2,Meera,23.0,None,NaN
3,Kabir,22.0,Delhi,91.0
4,None,20.0,Pune,85.0


## 1. Detect and summarize missingness

In [2]:
display(df.isna())
print('Missing count per column:')
display(df.isna().sum().to_frame('missing_count'))
print('Total missing cells:', int(df.isna().sum().sum()))
print('Missing percentage:')
display((df.isna().mean() * 100).round(1))

,name,age,city,score
0,False,False,False,False
1,False,True,False,False
2,False,False,True,True
3,False,False,False,False
4,True,False,False,False


Missing count per column:


,missing_count
name,1
age,1
city,1
score,1


Total missing cells: 4
Missing percentage:


name     20.0
age      20.0
city     20.0
score    20.0
dtype: float64

In [3]:
df[df['age'].isna()]

,name,age,city,score
1,Ravi,NaN,Pune,74.0


## 2. Drop rows or columns carefully

In [4]:
print('Drop any incomplete row:')
display(df.dropna())
print('Only require score to be present:')
display(df.dropna(subset=['score']))
print('Keep rows with at least three present values:')
display(df.dropna(thresh=3))

Drop any incomplete row:


,name,age,city,score
0,Asha,21.0,Indore,88.0
3,Kabir,22.0,Delhi,91.0


Only require score to be present:


,name,age,city,score
0,Asha,21.0,Indore,88.0
1,Ravi,NaN,Pune,74.0
3,Kabir,22.0,Delhi,91.0
4,None,20.0,Pune,85.0


Keep rows with at least three present values:


,name,age,city,score
0,Asha,21.0,Indore,88.0
1,Ravi,NaN,Pune,74.0
3,Kabir,22.0,Delhi,91.0
4,None,20.0,Pune,85.0


## 3. Fill with constants and statistics

In [5]:
filled = df.fillna({'name': 'Unknown', 'age': df['age'].median(), 'city': 'Unknown', 'score': df['score'].mean()})
filled

,name,age,city,score
0,Asha,21.0,Indore,88.0
1,Ravi,21.5,Pune,74.0
2,Meera,23.0,Unknown,84.5
3,Kabir,22.0,Delhi,91.0
4,Unknown,20.0,Pune,85.0


## 4. Forward fill and backward fill

In [6]:
series = pd.Series([10, np.nan, np.nan, 15, np.nan])
print('Original:', series.tolist())
print('Forward fill:', series.ffill().tolist())
print('Backward fill:', series.bfill().tolist())

Original: [10.0, nan, nan, 15.0, nan]
Forward fill: [10.0, 10.0, 10.0, 15.0, 15.0]
Backward fill: [10.0, 15.0, 15.0, 15.0, nan]


## 5. Replace placeholders and track missingness

In [7]:
placeholders = pd.DataFrame({'rating': ['5', 'N/A', '-', '4']})
placeholders = placeholders.replace(['N/A', '-'], pd.NA)
placeholders['rating_was_missing'] = placeholders['rating'].isna()
placeholders

,rating,rating_was_missing
0,5,False
1,<NA>,True
2,<NA>,True
3,4,False


## 6. Group-specific imputation with fallback

In [8]:
people = pd.DataFrame({'city': ['Indore', 'Indore', 'Pune', 'Pune', 'Delhi'], 'age': [20, np.nan, 30, np.nan, np.nan]})
city_median = people.groupby('city')['age'].transform('median')
global_median = people['age'].median()
people['age_filled'] = people['age'].fillna(city_median).fillna(global_median)
people

,city,age,age_filled
0,Indore,20.0,20.0
1,Indore,NaN,20.0
2,Pune,30.0,30.0
3,Pune,NaN,30.0
4,Delhi,NaN,25.0


## 7. Missingness indicator and before/after audit

In [9]:
raw = df.copy()
cleaned = raw.copy()
cleaned['score_was_missing'] = cleaned['score'].isna()
cleaned['age'] = cleaned['age'].fillna(cleaned['age'].median())
cleaned['city'] = cleaned['city'].fillna('Unknown')
print('Before cleaning:')
display(raw.isna().sum())
print('After selected cleaning:')
display(cleaned.isna().sum())
cleaned

Before cleaning:


name     1
age      1
city     1
score    1
dtype: int64

After selected cleaning:


name                 1
age                  0
city                 0
score                1
score_was_missing    0
dtype: int64

,name,age,city,score,score_was_missing
0,Asha,21.0,Indore,88.0,False
1,Ravi,21.5,Pune,74.0,False
2,Meera,23.0,Unknown,NaN,True
3,Kabir,22.0,Delhi,91.0,False
4,None,20.0,Pune,85.0,False


## Key takeaways

- Profile missingness before cleaning.
- Missing values are not necessarily zero.
- Use domain context to choose deletion or imputation.
- Preserve raw data and avoid leakage when building ML pipelines.